# Python Type Hints

**Goal:** Type hints, Protocol, generics, and mypy — the biggest shock is that hints are **completely ignored at runtime**.

**Prereq:** Complete [02_oop_patterns.ipynb](./02_oop_patterns.ipynb) first.

## The Big Shock: Hints Are Ignored at Runtime

**Go/TS comparison:** In Go, types are enforced at compile time — wrong type = won't compile. In TS, types are enforced at compile time but erased at runtime. In Python, type hints are **completely ignored at runtime** — they're documentation that tools like mypy can optionally check. `x: int = "hello"` runs without error.

In [ ]:
# This runs perfectly fine — Python doesn't care about the hint
x: int = "hello"
print(f"x = {x}, type = {type(x)}")  # x = hello, type = <class 'str'>

def add(a: int, b: int) -> int:
    return a + b

# Passing strings to a function "typed" for ints — no error!
result = add("foo", "bar")
print(f"result = {result}")  # foobar

### ✍️ In Your Own Words

How does Python's approach to types compare to Go and TypeScript? What's the trade-off? Write your answer here.

## Basic Type Hints

In [ ]:
# Function signatures
def greet(name: str) -> str:
    return f"Hello, {name}!"

# Variable annotations
count: int = 0
names: list[str] = ["Alice", "Bob"]
scores: dict[str, float] = {"Alice": 95.5, "Bob": 87.0}

print(greet("Kyle"))
print(f"Count: {count}, Names: {names}")

In [ ]:
# Optional and Union — for values that might be None
from typing import Optional

def find_index(items: list[str], target: str) -> Optional[int]:
    """Returns index or None if not found — like Go's comma-ok pattern"""
    try:
        return items.index(target)
    except ValueError:
        return None

result = find_index(["a", "b", "c"], "b")
print(f"Found at: {result}")  # 1

result = find_index(["a", "b", "c"], "z")
print(f"Not found: {result}")  # None

In [ ]:
# Experiment: Python 3.10+ union syntax — cleaner than Optional
# Optional[int] is just shorthand for int | None
def find_v2(items: list[str], target: str) -> int | None:
    try:
        return items.index(target)
    except ValueError:
        return None

# Union for multiple types
value: int | str = "hello"
value = 42  # Also fine
print(f"value = {value}")

### ✍️ In Your Own Words

What's the difference between `Optional[int]` and `int | None`? Write your answer here.

## Complex Types

In [ ]:
from typing import Callable, TypedDict

# Nested collections
matrix: list[list[int]] = [[1, 2], [3, 4]]
config: dict[str, list[int]] = {"ports": [8080, 8443]}

# Fixed-length tuple (like TS [string, number])
point: tuple[float, float] = (3.14, 2.71)

# Variable-length tuple
ids: tuple[int, ...] = (1, 2, 3, 4, 5)

# Function types (like Go's func(int, int) int)
operation: Callable[[int, int], int] = lambda a, b: a + b
print(f"3 + 4 = {operation(3, 4)}")

In [ ]:
# TypedDict — like TS interfaces for dict shapes
# or like a Go struct but still a dict at runtime

class UserProfile(TypedDict):
    name: str
    age: int
    email: str

# mypy would check that all keys are present with correct types
user: UserProfile = {
    "name": "Kyle",
    "age": 30,
    "email": "kyle@example.com"
}
print(user)
print(f"Still a regular dict: {type(user)}")

### ✍️ In Your Own Words

How does `TypedDict` compare to Go structs or TS interfaces? What's the key difference? Write your answer here.

## Protocol (Structural Typing)

**Go/TS comparison:** THIS is the one that feels like Go interfaces. `Protocol` defines **structural types** — if an object has the right methods, it satisfies the protocol. No explicit inheritance needed. It's duck typing with type checker support.

In [ ]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> str: ...

# These classes DON'T inherit from Drawable — they just have draw()
class Circle:
    def __init__(self, radius: float):
        self.radius = radius
    def draw(self) -> str:
        return f"○ (radius={self.radius})"

class Square:
    def __init__(self, side: float):
        self.side = side
    def draw(self) -> str:
        return f"□ (side={self.side})"

def render(shape: Drawable) -> None:
    """Accepts anything with a draw() method — like a Go interface"""
    print(f"  Drawing: {shape.draw()}")

render(Circle(5))
render(Square(3))

In [ ]:
# Experiment: a class that DOESN'T satisfy the Protocol
class Triangle:
    def __init__(self, base, height):
        self.base = base
        self.height = height
    # No draw() method!

# This works at RUNTIME (Python doesn't check types)
# But mypy would flag: "Triangle" is not compatible with "Drawable"
# render(Triangle(3, 4))  # Uncomment to see it run (but mypy would complain)

print("Protocol is only checked by mypy, not at runtime")
print("This is exactly like Go interfaces — structural, not nominal")

### ✍️ In Your Own Words

How is `Protocol` different from ABCs? When would you use each? Write your answer here.

## Generics

**Go/TS comparison:** Like Go generics (`func First[T any](items []T) T`) or TS generics (`function first<T>(items: T[]): T`).

In [ ]:
from typing import TypeVar

# TypeVar — the pre-3.12 way
T = TypeVar('T')

def first(items: list[T]) -> T:
    """Return first element — type-safe for any list type"""
    return items[0]

print(first([1, 2, 3]))         # int
print(first(["a", "b", "c"]))   # str

In [ ]:
# Python 3.12+ syntax — much cleaner, like Go/TS
# def first[T](items: list[T]) -> T:
#     return items[0]

# Bounded generics — like Go's type constraints
from typing import TypeVar, Protocol

class Comparable(Protocol):
    def __lt__(self, other) -> bool: ...

CT = TypeVar('CT', bound=Comparable)

def minimum(a: CT, b: CT) -> CT:
    """Works for any type that supports < comparison"""
    return a if a < b else b

print(f"min(3, 7) = {minimum(3, 7)}")
print(f"min('apple', 'banana') = {minimum('apple', 'banana')}")

### ✍️ In Your Own Words

How do Python generics compare to Go's type constraints? Write your answer here.

## Using mypy

mypy is the static type checker — it reads your hints and catches errors **before runtime**. It's like `go vet` or the TS compiler, but optional.

In [ ]:
# Let's create a file with type errors and run mypy on it
with open("/tmp/type_demo.py", "w") as f:
    f.write('''
def add(a: int, b: int) -> int:
    return a + b

# These would be flagged by mypy:
result: str = add(1, 2)  # Wrong: add returns int, not str
add("hello", "world")     # Wrong: str is not int
''')

print("File written. In a terminal, you'd run:")
print("  pip install mypy")
print("  mypy /tmp/type_demo.py")
print()
print("mypy would report:")
print('  error: Incompatible types in assignment (expression has type "int", variable has type "str")')
print('  error: Argument 1 to "add" has incompatible type "str"; expected "int"')

### ✍️ In Your Own Words

When would you use mypy in a real project? When would you skip it? Write your answer here.

## Recap

- ✅ Type hints are **ignored at runtime** — they're documentation for tools
- ✅ `Optional[X]` = `X | None` — for nullable values
- ✅ `TypedDict` for typed dict shapes (like TS interfaces)
- ✅ `Protocol` for **structural typing** — duck typing + type checker (like Go interfaces)
- ✅ `TypeVar` for generics (cleaner `[T]` syntax in 3.12+)
- ✅ `mypy` for static type checking — optional but valuable

**Next:** [05_data_processing.ipynb](./05_data_processing.ipynb)